# Model Comparison: GEV vs LP3 vs TCEV

This notebook provides a side-by-side comparison of the three distributions applied to the same Australian catchment data.

In [ ]:
import pandas as pd
import numpy as np
import arviz as az
import matplotlib.pyplot as plt
from flood_ffa.data.bom import load_ams, get_flow_series
from flood_ffa.gev.fit import fit_gev
from flood_ffa.lp3.fit import fit_lp3
from flood_ffa.tcev.fit import fit_tcev
from flood_ffa.compare import plot_comparison
from flood_ffa.gev.plots import gev_return_level
from flood_ffa.lp3.plots import lp3_return_level
from flood_ffa.tcev.plots import tcev_return_level, gev_cdf_np

## Load Data and Fit Models

In [ ]:
df = load_ams("../data/AMS.csv")
flows = get_flow_series(df)

print("Fitting GEV...")
idata_gev = fit_gev(flows)
print("Fitting LP3...")
idata_lp3 = fit_lp3(flows)
print("Fitting TCEV...")
idata_tcev = fit_tcev(flows)

## Visual Comparison

Overlaid frequency curves with 94% HDI bands. Observe the divergence in the upper tail (rare events).

In [ ]:
fig = plot_comparison(idata_gev, idata_lp3, idata_tcev, flows)
plt.show()

## Design Flood Estimates

Comparison of estimated flows at standard Annual Exceedance Probabilities (AEP).

In [ ]:
def get_summary_table(idata_list, names, aeps):
    results = []
    for idata, name in zip(idata_list, names):
        post = idata.posterior
        if name == "GEV":
            mu, sigma, xi = post["mu"].values.flatten(), post["sigma"].values.flatten(), post["xi"].values.flatten()
            rls = np.array([gev_return_level(mu[i], sigma[i], xi[i], aeps) for i in range(len(mu))])
        elif name == "LP3":
            mu, sigma, skew = post["mu"].values.flatten(), post["sigma"].values.flatten(), post["skew"].values.flatten()
            rls = np.array([lp3_return_level(mu[i], sigma[i], skew[i], aeps) for i in range(len(mu))])
        elif name == "TCEV":
            w, mu1, sigma1, xi1 = post["w"].values.flatten(), post["mu1"].values.flatten(), post["sigma1"].values.flatten(), post["xi1"].values.flatten()
            mu2, sigma2, xi2 = post["mu2"].values.flatten(), post["sigma2"].values.flatten(), post["xi2"].values.flatten()
            x_grid = np.linspace(0, 1000, 5000)
            rls = np.zeros((len(w), len(aeps)))
            for i in range(len(w)):
                p_target = 1 - aeps / 100.0
                c1, c2 = gev_cdf_np(x_grid, mu1[i], sigma1[i], xi1[i]), gev_cdf_np(x_grid, mu2[i], sigma2[i], xi2[i])
                mix = (1-w[i])*c1 + w[i]*c2
                rls[i, :] = np.interp(p_target, mix, x_grid)
        
        median = np.median(rls, axis=0)
        hdi = az.hdi(rls, hdi_prob=0.94)
        for j, aep in enumerate(aeps):
            results.append({
                "Model": name,
                "AEP (%)": aep,
                "Median (m3/s)": round(median[j], 1),
                "Lower 94% HDI": round(hdi[j, 0], 1),
                "Upper 94% HDI": round(hdi[j, 1], 1)
            })
    return pd.DataFrame(results)

aeps_to_check = np.array([10, 5, 2, 1, 0.5, 0.2, 0.05])
summary_df = get_summary_table([idata_gev, idata_lp3, idata_tcev], ["GEV", "LP3", "TCEV"], aeps_to_check)
summary_df.pivot(index="AEP (%)", columns="Model", values="Median (m3/s)").sort_index(ascending=False)

## Detailed Summary with Uncertainty

In [ ]:
summary_df